# Password Cracking with Rainbow Table

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:

import hashlib, pandas as pd, os, string, itertools
from tqdm.notebook import tqdm

#  Hash function
def hash_password(pw, algo='sha256'):
    h = hashlib.new(algo)
    h.update(pw.encode())
    return h.hexdigest()

#  Salted hash
def salted_hash(pw, salt=None, algo='sha256'):
    salt = salt or os.urandom(8).hex()
    return salt, hash_password(pw + salt, algo)

#  Rainbow Table Generator
def generate_rainbow(wordlist, algo='sha256'):
    return pd.DataFrame({
        'Password': wordlist,
        'Hash': [hash_password(p, algo) for p in wordlist]
    })

#  Crack hash using rainbow table
def crack_hash_rainbow(hash_val, rainbow_df):
    row = rainbow_df[rainbow_df['Hash'] == hash_val]
    return row['Password'].values[0] if not row.empty else None

#  Demo: passwords and hashes
passwords = ['123456', 'letmein', 'password', 'admin', 'qwerty']
rainbow = generate_rainbow(passwords)

#  Crack test hash
target_pw = 'letmein'
target_hash = hash_password(target_pw)
cracked_pw = crack_hash_rainbow(target_hash, rainbow)

print(f"Target password: {target_pw}")
print(f"Hashed: {target_hash}")
print(f"Cracked: {cracked_pw}")

# Tiny brute force
def brute_force(target_hash, charset=string.ascii_lowercase, max_len=3, algo='sha256'):
    for l in range(1, max_len+1):
        for p in itertools.product(charset, repeat=l):
            guess = ''.join(p)
            if hash_password(guess, algo) == target_hash:
                return guess
    return None

# Brute-force demo (only works with tiny passwords)
tiny_pw = 'abc'
tiny_hash = hash_password(tiny_pw)
brute_result = brute_force(tiny_hash)

print(f"Brute-forced: {brute_result}")
